# To run this colab, put the following cmd in a terminal:
# ```shell
# $ jupyter notebook
# ```

In [11]:
# import sys
# !{sys.executable} -m pip install igraph
# !{sys.executable} -m pip install pyomo
# !{sys.executable} -m pip install pynverse

In [1]:
import layer_lib
import numpy as np
import configs
import constants
import os
import shutil
import time
import pandas as pd
import importlib
import math
from decimal import Decimal
from pynverse import inversefunc
importlib.reload(configs)

<module 'configs' from '/Users/shuangfeng/Documents/git/P_24_Evacuation_Plan/python_codes/od_model_slacks_flow_iep_fixcap/configs.py'>

In [46]:
configs.BASELINE

False

In [31]:

def od_approach():
    solvable_od, solvable_od_demand, infessible_od, infessible_od_demand, graph, d_adj,cap,capacity_info = layer_lib.layer1()
    facilities = np.zeros(len(graph.vs), dtype=int)
    # Set the i-th element in PRESET_STATION to 1
    for i in configs.PRESET_STATION:
        facilities[i] = 1
    sorted_od = layer_lib.sort_od_to_hazard(solvable_od, solvable_od_demand, infessible_od, infessible_od_demand, graph)
    data = None
    pre_charger_usage = np.zeros(len(graph.vs), dtype=int)
    old_od_pairs = None
    old_demand_list = None
    # new_cap = cap
    count = 0
    for key,value in sorted_od.items():
        if value == []:
            continue
        else:
            # if hasattr(data, 'flow_size'):
            #     new_cap = cap - (data.flow_size).to_numpy()
                # new_cap=np.where(new_cap < 0, 0, new_cap)
            # print(pd.DataFrame(new_cap))
            value_arr = np.array(value)
            od_pairs = value_arr[:, 0:3]
            demand_list = value_arr[:, 3]
            if not old_od_pairs is None:
                od_pairs = np.concatenate((old_od_pairs, od_pairs), axis=0)
                demand_list = np.concatenate((old_demand_list, demand_list), axis=0)
            if data is not None:
                facilities = data.charging_demand_size.iloc[:, 0].to_numpy()
                pre_charger_usage = data.charging_demand_size.iloc[:, 1].to_numpy()
            print("start solve layer 3")
            print(od_pairs,demand_list)
            data = layer_lib.od_layer3(od_pairs, demand_list, graph, d_adj, facilities,pre_charger_usage=pre_charger_usage, cap=cap,capacity_info = capacity_info, data=data)
            count += 1
            if count >= 2:
                break
            print((data.od_plans.values()))
            # for od_pair, od_plan in data.od_plans.items():
            #     print(f"OD Pair: {od_pair}, Plan: {od_plan}")
            old_od_pairs = od_pairs
            old_demand_list = demand_list
    return data

In [14]:
importlib.reload(layer_lib)

<module 'layer_lib' from '/Users/shuangfeng/Documents/git/P_24_Evacuation_Plan/python_codes/od_model_slacks_flow_iep_fixcap/layer_lib.py'>

In [32]:
data = od_approach()

configs.PATH_CONFIG.RELATIVE_RAW_GRAPH_PATH: /Users/shuangfeng/Documents/git/P_24_Evacuation_Plan/python_codes/SiouxFalls_oneway_customOD2/SiouxFalls_oneway_customOD2.csv
configs.PATH_CONFIG.RELATIVE_OD_WITH_DEMAND_PATH: /Users/shuangfeng/Documents/git/P_24_Evacuation_Plan/python_codes/SiouxFalls_oneway_customOD2/SiouxFalls_oneway_customOD2_od_demand.csv
number of unconnected od pairs:  0
--------------------------------------------------
start solve layer 3
[[ 0 19  0]] [10000]
Layer3 solved
dict_values([      i     j  remaining energy  charged energy
0   0.0   2.0             122.0             0.0
1   2.0   3.0             119.0             0.0
2   3.0  10.0             117.0             0.0
3  10.0   9.0             115.0             0.0
4   9.0  16.0             113.0             0.0
5  16.0  18.0             111.0             0.0
6  18.0  19.0             110.0             0.0])
start solve layer 3
[[ 0 19  0]
 [ 2 19  0]] [10000 10000]
Layer3 solved


In [33]:
data

RunData(metadata=                      name         value
0  non-zero flow_size mean  10769.230769
1   non-zero flow_size std   2664.693550, flow_size=     0    1        2        3        4    5    6    7        8        9   ...  \
0   0.0  0.0  20000.0      0.0      0.0  0.0  0.0  0.0      0.0      0.0  ...   
1   0.0  0.0      0.0      0.0      0.0  0.0  0.0  0.0      0.0      0.0  ...   
2   0.0  0.0      0.0  30000.0      0.0  0.0  0.0  0.0      0.0      0.0  ...   
3   0.0  0.0      0.0      0.0  10000.0  0.0  0.0  0.0      0.0      0.0  ...   
4   0.0  0.0      0.0      0.0      0.0  0.0  0.0  0.0  10000.0      0.0  ...   
5   0.0  0.0      0.0      0.0      0.0  0.0  0.0  0.0      0.0      0.0  ...   
6   0.0  0.0      0.0      0.0      0.0  0.0  0.0  0.0      0.0      0.0  ...   
7   0.0  0.0      0.0      0.0      0.0  0.0  0.0  0.0      0.0      0.0  ...   
8   0.0  0.0      0.0      0.0      0.0  0.0  0.0  0.0      0.0  10000.0  ...   
9   0.0  0.0      0.0      0.0      0.0

In [49]:
key = list(data.od_plans.keys())

In [51]:
key[0]

(0, 19, 0)

In [52]:
df_data = data.od_plans[key[0]]


In [53]:
df_data

,i,j,remaining energy,charged energy
0,0.0,2.0,122.0,0.0
1,2.0,3.0,119.0,0.0
2,3.0,4.0,117.0,0.0
3,4.0,8.0,115.0,0.0
4,8.0,9.0,113.0,0.0
5,9.0,16.0,112.0,0.0
6,16.0,18.0,110.0,0.0
7,18.0,19.0,109.0,0.0


In [21]:
df_data.reset_index?

Signature:
df_data.reset_index(
    level: 'IndexLabel | None' = None,
    *,
    drop: 'bool' = False,
    inplace: 'bool' = False,
    col_level: 'Hashable' = 0,
    col_fill: 'Hashable' = '',
    allow_duplicates: 'bool | lib.NoDefault' = <no_default>,
    names: 'Hashable | Sequence[Hashable] | None' = None,
) -> 'DataFrame | None'
Docstring:
Reset the index, or a level of it.

Reset the index of the DataFrame, and use the default one instead.
If the DataFrame has a MultiIndex, this method can remove one or more
levels.

Parameters
----------
level : int, str, tuple, or list, default None
    Only remove the given levels from the index. Removes all levels by
    default.
drop : bool, default False
    Do not try to insert index into dataframe columns. This resets
    the index to the default integer index.
inplace : bool, default False
    Whether to modify the DataFrame rather than creating a new one.
col_level : int or str, default 0
    If the columns have multiple levels, deter

In [45]:
res = []
for item, df_data in data.od_plans.items():
    res.append(list(zip(df_data['i'], df_data['j'])))
print(res)

[[(0.0, 2.0), (2.0, 3.0), (3.0, 4.0), (4.0, 8.0), (8.0, 9.0), (9.0, 16.0), (16.0, 18.0), (18.0, 19.0)], [(2.0, 3.0), (3.0, 10.0), (10.0, 13.0), (13.0, 22.0), (22.0, 21.0), (21.0, 19.0)]]
